# 07 — Stage 2: Onset Models

Logistic regression models of civil war onset augmented by intervention expectations.

**Inputs**:
- `data/interim/cy_imputed_{1..5}.parquet` — CY baseline covariates (from nb01)
- `data/interim/cy_shadow_{cy}_{ud}.parquet` — shadow measure (from nb06)

**Outputs** (data saved to `results/` for nb09 to visualize):
- `results/stage2_fit.parquet`       — per-spec in-sample + OOS fit metrics
- `results/stage2_coefs.parquet`     — logit coefficients and SEs per spec
- `results/stage2_annual.parquet`    — annual predicted probabilities
- `results/stage2_proximate.parquet` — 5-year window around onset
- `results/stage2_vuong.parquet`     — pairwise Vuong test results

**Models**:
| Name          | Extended variables |
|---------------|--------------------|
| Baseline      | F&L-in-spirit covariates only |
| Entrants      | + E_gov_trim_asinh, E_opp_trim_asinh |
| Powers        | + E_major_gov/opp_trim_asinh |
| Neighbors     | + E_contig_gov/opp_trim_asinh |
| Coethnics     | + E_coethnic_gov/opp_trim_asinh |
| Rulers        | + E_colonial_gov/opp_trim_asinh |
| Rivals (bin)  | + E_rival_gov/opp_trim_asinh |
| Rivals (cts)  | + E_hostile_gov/opp_trim_asinh |
| Full          | all entrants + type-disaggregated (excl. one rivals variant) |

**Evaluation**: in-sample log-loss + AUC + PRL + Vuong test;
out-of-sample leave-one-onset-group-out log-loss + AUC + PRL.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"
RESULTS = ROOT / "results"
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)

## § 1  Load and merge data

In [2]:
# Baseline covariates (Fearon & Laitin in spirit, not letter)
# Time-varying covariates lagged one year to avoid post-onset contamination
BASELINE_VARS = [
    "polity2_lag", "lgdp_lag", "lpop_lag",
    "lmtnest", "ncontig", "oil",
    "nwstate", "instab_lag", "prior_war",
    "ethfrac", "relfrac",
    "year",   # controls for number-of-states trend
]


def _build_preperiod() -> pd.DataFrame:
    """Build 1945 polity2 and instab from raw V-Dem, matching nb01 logic.

    This avoids losing 1946 rows when computing polity2_lag and instab_lag
    via groupby.shift(1) on the 1946–2014 analysis panel.
    """
    import zipfile

    vdem_zip = ROOT / "data" / "raw" / "vdem" / "V-Dem-CY-FullOthers-v15_csv.zip"
    with zipfile.ZipFile(vdem_zip) as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
        vdem = pd.read_csv(
            z.open(csv_name),
            usecols=["COWcode", "year", "e_polity2", "e_pt_coup"],
            low_memory=False,
        )
    vdem = vdem[vdem["year"].between(1942, 1945)].copy()
    vdem = vdem.rename(columns={"COWcode": "ccode", "e_polity2": "polity2"})
    vdem["ccode"] = vdem["ccode"].astype("Int64").astype(str).str.zfill(3)

    # fix_ccode: 365 → 364 during Soviet era
    vdem.loc[vdem["ccode"] == "365", "ccode"] = "364"

    # Compute instab for 1945: |polity2_1945 − polity2_1942| ≥ 3, or coup
    vdem = vdem.sort_values(["ccode", "year"])
    vdem["polity2_lag3"] = vdem.groupby("ccode")["polity2"].shift(3)
    v45 = vdem[vdem["year"] == 1945].copy()
    delta3 = (v45["polity2"] - v45["polity2_lag3"]).abs()
    coup = v45["e_pt_coup"].fillna(0).astype(bool)
    v45["instab"] = ((delta3 >= 3) | coup).astype(float)
    v45.loc[v45["polity2"].isna() & ~coup, "instab"] = np.nan

    pre = v45[["ccode", "year", "polity2", "instab"]].copy()
    print(f"Pre-period 1945: {len(pre)} rows, "
          f"polity2 non-null: {pre['polity2'].notna().sum()}, "
          f"instab non-null: {pre['instab'].notna().sum()}")
    return pre


def load_analysis_data() -> pd.DataFrame:
    """Load and average shadow variables across 25 imputations."""
    # Average CY baseline covariates across 5 CY imputations
    cy_frames = [
        pd.read_parquet(INTERIM / f"cy_imputed_{i}.parquet")
        for i in range(1, 6)
    ]
    # Keep onset as binary (round after averaging to avoid fractional onsets)
    avg_cols = ["onset", "polity2", "lgdp_lag", "lpop_lag",
                "lmtnest", "ncontig", "oil", "nwstate", "instab",
                "prior_war", "ethfrac", "relfrac"]
    cy_avg = (
        pd.concat(cy_frames)
        .groupby(["ccode", "year"])[avg_cols]
        .mean()
        .reset_index()
    )
    cy_avg["onset"] = (cy_avg["onset"] >= 0.5).astype(int)

    # Append 1945 pre-period polity2/instab so shift(1) fills 1946 lags
    pre = _build_preperiod()
    cy_aug = pd.concat([pre, cy_avg], ignore_index=True)

    # Create lagged covariates: polity2 and instab
    cy_aug = cy_aug.sort_values(["ccode", "year"])
    cy_aug["polity2_lag"] = cy_aug.groupby("ccode")["polity2"].shift(1)
    cy_aug["instab_lag"]  = cy_aug.groupby("ccode")["instab"].shift(1)

    # Drop pre-period rows (keep only 1946+)
    cy_avg = cy_aug[cy_aug["year"] >= 1946].copy()

    n_lag_miss = cy_avg["polity2_lag"].isna().sum()
    print(f"polity2_lag still missing: {n_lag_miss} rows "
          f"(countries entering after 1946 or no 1945 Polity data)")

    # Average shadow variables across 25 imputations.
    sh_frames = []
    for cy in range(1, 6):
        for ud in range(1, 6):
            sh_frames.append(
                pd.read_parquet(INTERIM / f"cy_shadow_{cy}_{ud}.parquet")
            )
    sh_all = pd.concat(sh_frames, ignore_index=True)
    shadow_cols = [c for c in sh_all.columns
                   if c.startswith("E_") or c == "n_B"]
    shadow_avg = (
        sh_all.groupby(["ccode", "year"])[shadow_cols]
        .mean()
        .reset_index()
    )

    merged = cy_avg.merge(shadow_avg, on=["ccode", "year"], how="left")
    print(f"Shadow columns: {len(shadow_cols)}")
    return merged


data = load_analysis_data()
print(f"Analysis dataset: {len(data):,} country-years, "
      f"{data['onset'].sum():.0f} onsets, "
      f"{data.shape[1]} columns")

# Identify onset events for leave-one-onset-out CV.
# Each onset spell gets a unique group; non-onset rows get per-country groups.
data["onset_id"] = (
    data.sort_values(["ccode", "year"])
    .groupby("ccode")["onset"]
    .transform(lambda s: (s.cumsum() * s).replace(0, np.nan))
)
max_onset_id = data["onset_id"].max()
data["cv_group"] = data["onset_id"].fillna(
    data.groupby("ccode").ngroup() + max_onset_id + 1
)


Pre-period 1945: 149 rows, polity2 non-null: 67, instab non-null: 67
polity2_lag still missing: 105 rows (countries entering after 1946 or no 1945 Polity data)


Shadow columns: 73
Analysis dataset: 8,897 country-years, 191 onsets, 89 columns


## § 2  Model specifications

In [3]:
def make_specs(data: pd.DataFrame) -> dict[str, list[str]]:
    """Return model specifications as {name: covariate_list}."""
    base = BASELINE_VARS

    # Shadow variables: trimmed + asinh for each type
    entrants  = ["E_gov_trim_asinh", "E_opp_trim_asinh"]
    powers    = ["E_major_gov_trim_asinh", "E_major_opp_trim_asinh"]
    neighbors = ["E_contig_gov_trim_asinh", "E_contig_opp_trim_asinh"]
    coethnics = ["E_coethnic_gov_trim_asinh", "E_coethnic_opp_trim_asinh"]
    rulers    = ["E_colonial_gov_trim_asinh", "E_colonial_opp_trim_asinh"]
    rivals_b  = ["E_rival_gov_trim_asinh", "E_rival_opp_trim_asinh"]
    rivals_c  = ["E_hostile_gov_trim_asinh", "E_hostile_opp_trim_asinh"]
    doe       = ["E_doe_gov_trim_asinh", "E_doe_opp_trim_asinh"]

    # Each extended spec = base + aggregate entrants + type-specific.
    # The type-specific variables test whether that intervener characteristic
    # carries additional weight beyond the aggregate shadow signal.
    specs = {
        "Baseline":      base,
        "Entrants":      base + entrants,
        "Powers":        base + entrants + powers,
        "Neighbors":     base + entrants + neighbors,
        "Coethnics":     base + entrants + coethnics,
        "Rulers":        base + entrants + rulers,
        "Rivals (bin)":  base + entrants + rivals_b,
        "Rivals (cts)":  base + entrants + rivals_c,
        "DOE":           base + entrants + doe,
        # Full: entrants + all type-disagg (use continuous rivals to avoid
        # collinearity with binary rivals — both slice the same variation)
        "Full":          base + entrants + powers + neighbors
                         + coethnics + rulers + rivals_c + doe,
    }

    # Verify all variables exist in the data
    for name, covars in specs.items():
        missing = [c for c in covars if c not in data.columns]
        if missing:
            print(f"WARNING: {name} missing columns: {missing}")

    return specs


SPECS = make_specs(data)
print(f"\n{len(SPECS)} specifications:")
for name, covars in SPECS.items():
    print(f"  {name:16s}: {len(covars)} vars")


10 specifications:
  Baseline        : 12 vars
  Entrants        : 14 vars
  Powers          : 16 vars
  Neighbors       : 16 vars
  Coethnics       : 16 vars
  Rulers          : 16 vars
  Rivals (bin)    : 16 vars
  Rivals (cts)    : 16 vars
  DOE             : 16 vars
  Full            : 26 vars


## § 3  Fit models and evaluate

In [4]:
def prl(y: np.ndarray, y_hat: np.ndarray) -> float:
    """Proportional reduction in log-loss vs. null."""
    null_p  = np.clip(y.mean(), 1e-9, 1 - 1e-9)
    null_ll = -(y * np.log(null_p) + (1 - y) * np.log(1 - null_p)).mean()
    model_ll = log_loss(y, y_hat)
    return (null_ll - model_ll) / null_ll


def _fit_logit(y, X, maxiter=1000):
    """Fit logit with BFGS fallback if Newton Hessian is singular."""
    try:
        return sm.Logit(y, X).fit(disp=False, maxiter=maxiter)
    except np.linalg.LinAlgError:
        return sm.Logit(y, X).fit(disp=False, method="bfgs", maxiter=maxiter)


def fit_and_eval(spec_name: str, covars: list[str]) -> dict:
    sub = data[["onset", "cv_group"] + covars].dropna().copy()
    y   = sub["onset"].values
    X   = sm.add_constant(sub[covars].values)

    # In-sample fit
    logit = _fit_logit(y, X)
    y_hat = logit.predict(X)
    is_ll  = log_loss(y, y_hat)
    is_auc = roc_auc_score(y, y_hat)
    is_prl = prl(y, y_hat)

    # Out-of-fold: leave-one-onset-group-out
    groups = sub["cv_group"].values
    logo   = LeaveOneGroupOut()
    oof_preds = np.zeros(len(y))
    for tr_idx, val_idx in logo.split(X, y, groups):
        try:
            m_fold = _fit_logit(y[tr_idx], X[tr_idx], maxiter=500)
            oof_preds[val_idx] = m_fold.predict(X[val_idx])
        except Exception:
            oof_preds[val_idx] = y[tr_idx].mean()
    oof_clipped = np.clip(oof_preds, 1e-9, 1 - 1e-9)
    oos_ll  = log_loss(y, oof_clipped)
    oos_auc = roc_auc_score(y, oof_preds)
    oos_prl = prl(y, oof_clipped)

    # Extract coefficients and SEs
    coef_names = ["const"] + covars
    coefs = pd.DataFrame({
        "model":    spec_name,
        "variable": coef_names,
        "coef":     logit.params,
        "se":       logit.bse,
        "z":        logit.tvalues,
        "p":        logit.pvalues,
    })

    return {
        "fit": {
            "model":   spec_name,
            "n":       len(y),
            "n_onset": int(y.sum()),
            "n_vars":  len(covars),
            "is_ll":   round(is_ll, 4),
            "is_auc":  round(is_auc, 4),
            "is_prl":  round(is_prl, 4),
            "oos_ll":  round(oos_ll, 4),
            "oos_auc": round(oos_auc, 4),
            "oos_prl": round(oos_prl, 4),
        },
        "coefs":  coefs,
        "logit":  logit,
        "oof":    oof_preds,
        "y":      y,
    }


print("Fitting models...")
results = {}
for name, covars in SPECS.items():
    print(f"  {name}...", end=" ", flush=True)
    results[name] = fit_and_eval(name, covars)
    r = results[name]["fit"]
    print(f"IS PRL={r['is_prl']:.3f} AUC={r['is_auc']:.3f}  "
          f"OOS PRL={r['oos_prl']:.3f} AUC={r['oos_auc']:.3f}")

# ── Save fit metrics ───────────────────────────────────────────────────────
fit_df = pd.DataFrame([r["fit"] for r in results.values()])
fit_df.to_parquet(RESULTS / "stage2_fit.parquet", index=False)
print(f"\nFit metrics saved: {RESULTS / 'stage2_fit.parquet'}")
print(fit_df.to_string(index=False))

# ── Save coefficients ─────────────────────────────────────────────────────
coefs_df = pd.concat([r["coefs"] for r in results.values()], ignore_index=True)
coefs_df.to_parquet(RESULTS / "stage2_coefs.parquet", index=False)
print(f"\nCoefficients saved: {len(coefs_df)} rows")

Fitting models...
  Baseline... 

IS PRL=0.101 AUC=0.771  OOS PRL=-0.010 AUC=0.650
  Entrants... 

IS PRL=0.121 AUC=0.793  OOS PRL=0.005 AUC=0.679
  Powers... 

IS PRL=0.139 AUC=0.804  OOS PRL=0.012 AUC=0.691
  Neighbors... 

IS PRL=0.135 AUC=0.808  OOS PRL=0.011 AUC=0.696
  Coethnics... 

...

IS PRL=0.122 AUC=0.793  OOS PRL=-0.014 AUC=0.676
  Rivals (bin)... 

IS PRL=0.133 AUC=0.804  OOS PRL=0.004 AUC=0.688
  Rivals (cts)... 

IS PRL=0.136 AUC=0.808  OOS PRL=0.011 AUC=0.692
  DOE... 

IS PRL=0.122 AUC=0.793  OOS PRL=0.002 AUC=0.678
  Full... 

IS PRL=0.177 AUC=0.840  OOS PRL=-0.021 AUC=0.725

Fit metrics saved: /Users/rjc/portfolio/shadow/results/stage2_fit.parquet
       model    n  n_onset  n_vars  is_ll  is_auc  is_prl  oos_ll  oos_auc  oos_prl
    Baseline 8792      184      12 0.0914  0.7710  0.1008  0.1027   0.6496  -0.0101
    Entrants 8792      184      14 0.0893  0.7932  0.1215  0.1012   0.6794   0.0046
      Powers 8792      184      16 0.0875  0.8042  0.1387  0.1004   0.6906   0.0117
   Neighbors 8792      184      16 0.0879  0.8080  0.1350  0.1005   0.6959   0.0111
   Coethnics 8792      184      16 0.0888  0.8005  0.1264  0.1014   0.6807   0.0025
      Rulers 8792      184      16 0.0893  0.7932  0.1218  0.1031   0.6757  -0.0142
Rivals (bin) 8792      184      16 0.0881  0.8036  0.1333  0.1012   0.6876   0.0040
Rivals (cts) 8792      184      16 0.0879  0.8084  0.1356  0.1005   0.6917   0.0113
         DOE 8792      184      16 0.0892  0.7934  0.1220  0.1014   0.6776   0.0020
        Full 8792      184      26 0

## § 4  Vuong test (Baseline vs. Entrants)

In [5]:
def vuong_test(model_A_name: str, model_B_name: str) -> dict:
    """Non-nested Vuong test comparing two logit models on their common sample.

    Refits both models on the intersection of their non-missing rows so that
    predictions are on the same observations.  Returns dict with z, p, n.
    z > 0 favours A.
    """
    covars_A = SPECS[model_A_name]
    covars_B = SPECS[model_B_name]

    mask_A = data[["onset"] + covars_A].notna().all(axis=1)
    mask_B = data[["onset"] + covars_B].notna().all(axis=1)
    common = data.index[mask_A & mask_B]

    sub = data.loc[common].copy()
    y   = sub["onset"].values

    X_A = sm.add_constant(sub[covars_A].values)
    X_B = sm.add_constant(sub[covars_B].values)

    m_A = sm.Logit(y, X_A).fit(disp=False, maxiter=1000)
    m_B = sm.Logit(y, X_B).fit(disp=False, maxiter=1000)

    p_A = np.clip(m_A.predict(X_A), 1e-9, 1 - 1e-9)
    p_B = np.clip(m_B.predict(X_B), 1e-9, 1 - 1e-9)

    ll_A = y * np.log(p_A) + (1 - y) * np.log(1 - p_A)
    ll_B = y * np.log(p_B) + (1 - y) * np.log(1 - p_B)
    m    = ll_A - ll_B
    n    = len(y)
    z    = np.sqrt(n) * m.mean() / m.std()
    p    = 2 * (1 - norm.cdf(abs(z)))
    return {"model_A": model_A_name, "model_B": model_B_name,
            "z": round(z, 3), "p": round(p, 4), "n": n}


# Pairwise Vuong tests: each extended model vs Baseline
vuong_rows = []
extended = [s for s in SPECS if s != "Baseline"]
for name in extended:
    v = vuong_test(name, "Baseline")
    star = "***" if v["p"] < 0.001 else "**" if v["p"] < 0.01 else "*" if v["p"] < 0.05 else ""
    print(f"  {name:16s} vs Baseline: z={v['z']:+.3f}, p={v['p']:.4f} {star}")
    vuong_rows.append(v)

# Also compare Rivals (bin) vs Rivals (cts) directly
v = vuong_test("Rivals (bin)", "Rivals (cts)")
print(f"\n  Rivals (bin) vs Rivals (cts): z={v['z']:+.3f}, p={v['p']:.4f}")
vuong_rows.append(v)

vuong_df = pd.DataFrame(vuong_rows)
vuong_df.to_parquet(RESULTS / "stage2_vuong.parquet", index=False)
print(f"\nVuong tests saved: {len(vuong_df)} comparisons")

  Entrants         vs Baseline: z=+2.837, p=0.0046 **
  Powers           vs Baseline: z=+3.529, p=0.0004 ***
  Neighbors        vs Baseline: z=+3.698, p=0.0002 ***
  Coethnics        vs Baseline: z=+3.268, p=0.0011 **
  Rulers           vs Baseline: z=+2.861, p=0.0042 **
  Rivals (bin)     vs Baseline: z=+3.903, p=0.0001 ***
  Rivals (cts)     vs Baseline: z=+3.796, p=0.0001 ***
  DOE              vs Baseline: z=+2.963, p=0.0030 **
  Full             vs Baseline: z=+5.401, p=0.0000 ***

  Rivals (bin) vs Rivals (cts): z=-0.957, p=0.3383

Vuong tests saved: 10 comparisons


## § 5  Cold War interaction analysis

Interact shadow variables with Cold War indicator (year ≤ 1990).
Tests whether intervention expectations have different effects during
vs. after the Cold War — motivated by the superpower-proxy-war hypothesis.

In [6]:
# Cold War interaction: Entrants model with CW × shadow interactions
data["cold_war"] = (data["year"] <= 1990).astype(int)

cw_covars = (
    BASELINE_VARS
    + ["E_gov_trim_asinh", "E_opp_trim_asinh"]
    + ["cold_war"]
    + ["cw_E_gov", "cw_E_opp"]
)
data["cw_E_gov"] = data["cold_war"] * data["E_gov_trim_asinh"]
data["cw_E_opp"] = data["cold_war"] * data["E_opp_trim_asinh"]

sub_cw = data[["onset"] + cw_covars].dropna().copy()
y_cw = sub_cw["onset"].values
X_cw = sm.add_constant(sub_cw[cw_covars].values)
logit_cw = _fit_logit(y_cw, X_cw)

cw_names = ["const"] + cw_covars
cw_coefs = pd.DataFrame({
    "variable": cw_names,
    "coef": logit_cw.params,
    "se": logit_cw.bse,
    "z": logit_cw.tvalues,
    "p": logit_cw.pvalues,
})

print("Cold War interaction model:")
print(cw_coefs[cw_coefs["variable"].str.contains("E_|cold")].to_string(index=False))
print(f"\nCold War obs: {(sub_cw['cold_war'] == 1).sum()}, "
      f"Post-CW: {(sub_cw['cold_war'] == 0).sum()}")

# Save
cw_coefs["model"] = "CW Interaction"
cw_coefs.to_parquet(RESULTS / "stage2_cw_coefs.parquet", index=False)

Cold War interaction model:
        variable      coef       se         z        p
E_gov_trim_asinh -1.815139 0.561650 -3.231796 0.001230
E_opp_trim_asinh  0.872830 0.691645  1.261961 0.206963
        cold_war  0.279413 0.437011  0.639372 0.522581
        cw_E_gov -0.377946 0.671797 -0.562589 0.573715
        cw_E_opp  0.675195 0.802557  0.841304 0.400178

Cold War obs: 5045, Post-CW: 3747


## § 6  Annual and proximate-year diagnostics

Save predicted probabilities by year (for temporal performance plot)
and around onset events (for proximate-year analysis).

In [7]:
# ── Annual predicted probabilities (Baseline vs Entrants) ──────────────────
annual_rows = []
for spec_name in ["Baseline", "Entrants"]:
    covars = SPECS[spec_name]
    # Deduplicate columns (year appears in both explicit list and covars)
    cols = list(dict.fromkeys(["onset", "year", "ccode"] + covars))
    sub = data[cols].dropna().copy()
    y = sub["onset"].values
    X = sm.add_constant(sub[covars].values)
    logit = _fit_logit(y, X)
    sub["p_hat"] = logit.predict(X)
    by_year = sub.groupby("year").agg(
        onset_rate=("onset", "mean"),
        mean_p=("p_hat", "mean"),
        n=("onset", "count"),
        n_onset=("onset", "sum"),
    ).reset_index()
    by_year["model"] = spec_name
    annual_rows.append(by_year)

annual_df = pd.concat(annual_rows, ignore_index=True)
annual_df.to_parquet(RESULTS / "stage2_annual.parquet", index=False)
print(f"Annual predictions saved: {len(annual_df)} rows "
      f"({annual_df['year'].min()}-{annual_df['year'].max()})")

# ── Proximate-year analysis (5-year window around onset) ───────────────────
# For each onset event, extract predicted probabilities at t-2...t+2
onset_events = data.loc[data["onset"] == 1, ["ccode", "year"]].copy()
print(f"\nOnset events: {len(onset_events)}")

prox_rows = []
for spec_name in SPECS:
    covars = SPECS[spec_name]
    cols = list(dict.fromkeys(["onset", "year", "ccode"] + covars))
    sub = data[cols].dropna().copy()
    y = sub["onset"].values
    X = sm.add_constant(sub[covars].values)
    logit = _fit_logit(y, X)
    sub["p_hat"] = logit.predict(X)

    for _, row in onset_events.iterrows():
        cc, yr = row["ccode"], row["year"]
        for dt in range(-2, 3):
            match = sub[(sub["ccode"] == cc) & (sub["year"] == yr + dt)]
            if len(match) == 1:
                prox_rows.append({
                    "model": spec_name,
                    "ccode": cc, "onset_year": yr,
                    "dt": dt, "year": yr + dt,
                    "p_hat": match["p_hat"].values[0],
                    "onset": match["onset"].values[0],
                })

prox_df = pd.DataFrame(prox_rows)
prox_df.to_parquet(RESULTS / "stage2_proximate.parquet", index=False)

# Print mean p_hat by dt for Baseline vs Entrants
for spec in ["Baseline", "Entrants"]:
    sub = prox_df[prox_df["model"] == spec]
    means = sub.groupby("dt")["p_hat"].mean()
    print(f"\n{spec} mean p_hat around onset:")
    for dt in range(-2, 3):
        print(f"  t{dt:+d}: {means.get(dt, float('nan')):.4f}")

print(f"\nProximate-year data saved: {len(prox_df)} rows")

Annual predictions saved: 138 rows (1946-2014)

Onset events: 191



Baseline mean p_hat around onset:
  t-2: 0.0416
  t-1: 0.0423
  t+0: 0.0465
  t+1: 0.0655
  t+2: 0.0603

Entrants mean p_hat around onset:
  t-2: 0.0434
  t-1: 0.0460
  t+0: 0.0520
  t+1: 0.0676
  t+2: 0.0631

Proximate-year data saved: 8970 rows


## § 7  Population attenuation check

The key theoretical prediction: controlling for intervention expectations
should attenuate the population coefficient, because larger countries attract
more potential interveners (confounding population → onset).

In [8]:
# Population coefficient across specifications
pop_coefs = coefs_df[coefs_df["variable"] == "lpop_lag"][
    ["model", "coef", "se", "z", "p"]
].copy()
pop_coefs = pop_coefs.set_index("model")

print("Population (lpop_lag) coefficient across specifications:")
print(pop_coefs.to_string())

baseline_pop = pop_coefs.loc["Baseline", "coef"]
print(f"\nBaseline lpop_lag coef: {baseline_pop:.4f}")
for model in pop_coefs.index:
    if model != "Baseline":
        pct_change = (pop_coefs.loc[model, "coef"] - baseline_pop) / abs(baseline_pop) * 100
        print(f"  {model:16s}: {pop_coefs.loc[model, 'coef']:+.4f} "
              f"({pct_change:+.1f}% vs Baseline)")

# ── Final summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Stage 2 complete. Output files:")
for f in sorted(RESULTS.glob("stage2_*.parquet")):
    print(f"  {f.name}")
print("=" * 60)

Population (lpop_lag) coefficient across specifications:
                  coef        se         z         p
model                                               
Baseline      0.109763  0.061268  1.791509  0.073212
Entrants      0.057677  0.067734  0.851511  0.394486
Powers        0.093632  0.068896  1.359024  0.174139
Neighbors     0.036679  0.069110  0.530742  0.595597
Coethnics     0.062555  0.068735  0.910083  0.362779
Rulers        0.049072  0.068857  0.712666  0.476053
Rivals (bin)  0.010046  0.070776  0.141937  0.887130
Rivals (cts) -0.005174  0.071371 -0.072492  0.942210
DOE           0.055678  0.068104  0.817538  0.413621
Full          0.018358  0.078160  0.234877  0.814304

Baseline lpop_lag coef: 0.1098
  Entrants        : +0.0577 (-47.5% vs Baseline)
  Powers          : +0.0936 (-14.7% vs Baseline)
  Neighbors       : +0.0367 (-66.6% vs Baseline)
  Coethnics       : +0.0626 (-43.0% vs Baseline)
  Rulers          : +0.0491 (-55.3% vs Baseline)
  Rivals (bin)    : +0.0100 (-

## § 8  Country Fixed Effects

The strong test: does *within-country* variation in the shadow predict
within-country variation in onset, eliminating all cross-sectional confounding?

- **Conditional logit** (Chamberlain): conditions out the country fixed effect.
  Only countries with at least one onset *and* at least one non-onset year contribute.
  Time-invariant covariates (terrain, ethnic frac, etc.) are not estimable and are dropped.
- **LPM with country dummies** (robustness): linear probability model with country FEs
  and cluster-robust SEs at the country level. Avoids the incidental parameters problem.

In [9]:
from statsmodels.discrete.conditional_models import ConditionalLogit
import statsmodels.formula.api as smf

print("=" * 60)
print("§ 8  Country Fixed Effects")
print("=" * 60)

# Time-varying covariates only (time-invariant absorbed by country FE)
FE_TIME_VARS = [
    "polity2_lag", "lgdp_lag", "lpop_lag",
    "nwstate", "instab_lag", "prior_war",
    "year",
]
SHADOW_VARS = ["E_gov_trim_asinh", "E_opp_trim_asinh"]

fe_specs = {
    "FE Baseline": FE_TIME_VARS,
    "FE Entrants": FE_TIME_VARS + SHADOW_VARS,
}

# ── Conditional Logit (Chamberlain) ────────────────────────────────────────
print("\n── Conditional Logit ──")

fe_results = {}
for name, covars in fe_specs.items():
    sub = data[["onset", "ccode"] + covars].dropna().copy()

    # Only keep countries with variation in onset (clogit requirement)
    onset_by_cc = sub.groupby("ccode")["onset"].agg(["sum", "count"])
    varying = onset_by_cc[
        (onset_by_cc["sum"] > 0) & (onset_by_cc["sum"] < onset_by_cc["count"])
    ].index
    sub_v = sub[sub["ccode"].isin(varying)].copy()

    y = sub_v["onset"].values
    X = sub_v[covars].astype(float).values
    groups = sub_v["ccode"].values

    clogit = ConditionalLogit(y, X, groups=groups)
    res = clogit.fit(disp=False)

    n_cc = len(varying)
    n_onset = int(y.sum())

    print(f"\n{name}:")
    print(f"  N = {len(y):,}, Countries = {n_cc}, Onsets = {n_onset}")
    for i, var in enumerate(covars):
        sig = ""
        if res.pvalues[i] < 0.001: sig = " ***"
        elif res.pvalues[i] < 0.01: sig = " **"
        elif res.pvalues[i] < 0.05: sig = " *"
        print(f"  {var:25s}: {res.params[i]:+.4f} "
              f"(SE={res.bse[i]:.4f}, p={res.pvalues[i]:.4f}){sig}")

    fe_results[name] = {
        "res": res, "covars": covars,
        "n": len(y), "n_cc": n_cc, "n_onset": n_onset,
    }

# ── LPM with country dummies (robustness) ─────────────────────────────────
print("\n\n── LPM with country FEs (cluster-robust SEs) ──")

for name, covars in fe_specs.items():
    sub = data[["onset", "ccode"] + covars].dropna().copy()
    formula = "onset ~ " + " + ".join(covars) + " + C(ccode)"
    lpm = smf.ols(formula, data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["ccode"]}
    )

    print(f"\n{name}:")
    print(f"  N = {len(sub):,}, Countries = {sub['ccode'].nunique()}, "
          f"Onsets = {int(sub['onset'].sum())}, R² = {lpm.rsquared:.4f}")
    for var in covars:
        coef = lpm.params[var]
        se = lpm.bse[var]
        p = lpm.pvalues[var]
        sig = ""
        if p < 0.001: sig = " ***"
        elif p < 0.01: sig = " **"
        elif p < 0.05: sig = " *"
        print(f"  {var:25s}: {coef:+.6f} (SE={se:.6f}, p={p:.4f}){sig}")

# ── Save FE coefficients ──────────────────────────────────────────────────
fe_coef_rows = []
for name in fe_specs:
    r = fe_results[name]
    for i, var in enumerate(r["covars"]):
        fe_coef_rows.append({
            "model": name, "variable": var,
            "coef": r["res"].params[i], "se": r["res"].bse[i],
            "z": r["res"].tvalues[i], "p": r["res"].pvalues[i],
        })
fe_coef_df = pd.DataFrame(fe_coef_rows)
fe_coef_df.to_parquet(RESULTS / "stage2_fe_coefs.parquet", index=False)
print(f"\nFE coefficients saved: {len(fe_coef_df)} rows")

§ 8  Country Fixed Effects

── Conditional Logit ──



FE Baseline:
  N = 3,750, Countries = 71, Onsets = 184
  polity2_lag              : -0.0321 (SE=0.0191, p=0.0926)
  lgdp_lag                 : -0.4742 (SE=0.2588, p=0.0669)
  lpop_lag                 : +0.7510 (SE=0.6807, p=0.2699)
  nwstate                  : +0.9963 (SE=0.3776, p=0.0083) **
  instab_lag               : +0.6364 (SE=0.1938, p=0.0010) **
  prior_war                : -0.2646 (SE=0.2006, p=0.1872)
  year                     : -0.0057 (SE=0.0185, p=0.7586)



FE Entrants:
  N = 3,750, Countries = 71, Onsets = 184
  polity2_lag              : -0.0382 (SE=0.0193, p=0.0473) *
  lgdp_lag                 : -0.2423 (SE=0.2680, p=0.3660)
  lpop_lag                 : +0.4747 (SE=0.7089, p=0.5031)
  nwstate                  : +1.2537 (SE=0.3864, p=0.0012) **
  instab_lag               : +0.6189 (SE=0.1956, p=0.0016) **
  prior_war                : -0.3493 (SE=0.2033, p=0.0857)
  year                     : +0.0023 (SE=0.0192, p=0.9052)
  E_gov_trim_asinh         : -2.3937 (SE=0.5343, p=0.0000) ***
  E_opp_trim_asinh         : +1.6892 (SE=0.5350, p=0.0016) **


── LPM with country FEs (cluster-robust SEs) ──

FE Baseline:
  N = 8,792, Countries = 171, Onsets = 184, R² = 0.0582
  polity2_lag              : -0.000582 (SE=0.000377, p=0.1221)
  lgdp_lag                 : -0.010048 (SE=0.007202, p=0.1630)
  lpop_lag                 : +0.001985 (SE=0.008760, p=0.8208)
  nwstate                  : +0.026367 (SE=0.015507, p=0.0891)
  instab_lag              


FE Entrants:
  N = 8,792, Countries = 171, Onsets = 184, R² = 0.0612
  polity2_lag              : -0.000686 (SE=0.000410, p=0.0942)
  lgdp_lag                 : -0.007072 (SE=0.007562, p=0.3497)
  lpop_lag                 : -0.000342 (SE=0.010511, p=0.9740)
  nwstate                  : +0.034040 (SE=0.016246, p=0.0361) *
  instab_lag               : +0.014929 (SE=0.006605, p=0.0238) *
  prior_war                : -0.013299 (SE=0.014865, p=0.3710)
  year                     : +0.000416 (SE=0.000334, p=0.2127)
  E_gov_trim_asinh         : -0.051525 (SE=0.015337, p=0.0008) ***
  E_opp_trim_asinh         : +0.040066 (SE=0.016454, p=0.0149) *

FE coefficients saved: 16 rows


## § 9  T×P Bootstrap (Knox, Lucas & Cho 2022)

The shadow variables are **generated regressors**: predicted from a Stage 1 classifier
rather than directly observed.  Standard logistic-regression SEs condition on the
Stage 1 model as fixed, ignoring measurement-stage uncertainty.  This produces
anticonservative (too-small) standard errors and p-values.

Following Knox, Lucas & Cho (2022, §4.2), we propagate uncertainty from *both* stages:

- **T = 25 measurement draws** — each `cy_shadow_{cy}_{ud}.parquet` is a different
  shadow measure, trained on a different multiply-imputed dataset.  These approximate
  the spread of measurement models consistent with the data.
- **P = 200 bootstrap replications per draw** — pairs cluster bootstrap resampling
  countries with replacement.  These approximate primary-analysis uncertainty
  conditional on a given shadow measure.

The full T×P = 5,000 coefficient vectors are pooled.  Standard errors are the SD
of this pooled distribution; confidence intervals are the 2.5th and 97.5th percentiles.
This captures uncertainty from both the measurement model (which shadow could have been
learned?) and the primary analysis (which countries happened to be in the sample?).

In [10]:
# ── T×P Bootstrap following KLC 2022 §4.2 ──────────────────────────────────
# T = 25 measurement draws × P = 200 cluster-bootstrap reps = 5,000 total

T_DRAWS = 25
P_REPS  = 200

rng = np.random.default_rng(seed=20260303)

# Pre-load baseline covariates (averaged across CY imputations, as in §1)
# We reuse the `data` frame for everything except the shadow columns.
# Deduplicate: "year" appears in both the explicit list and BASELINE_VARS
baseline_cols = list(dict.fromkeys(
    ["ccode", "year", "onset", "cv_group"] + BASELINE_VARS
))
base_df = data[baseline_cols].copy()

# Only bootstrap the Entrants specification (the primary result)
entrants_shadow = ["E_gov_trim_asinh", "E_opp_trim_asinh"]
entrants_covars = BASELINE_VARS + entrants_shadow
coef_names = ["const"] + entrants_covars

print(f"T×P bootstrap: {T_DRAWS} draws × {P_REPS} reps = "
      f"{T_DRAWS * P_REPS:,} total")
print(f"Covariates: {len(entrants_covars)}")

all_coefs = []
draw_labels = []  # track which draw each coef vector came from
draw_idx = 0

for cy in range(1, 6):
    for ud in range(1, 6):
        draw_idx += 1
        draw_key = f"{cy}_{ud}"

        # Load this specific shadow draw
        sh = pd.read_parquet(
            INTERIM / f"cy_shadow_{cy}_{ud}.parquet",
            columns=["ccode", "year"] + entrants_shadow,
        )

        # Merge with baseline
        df_t = base_df.merge(sh, on=["ccode", "year"], how="left")
        sub_t = df_t[list(dict.fromkeys(
            ["onset", "cv_group", "ccode"] + entrants_covars
        ))].dropna()

        y_full = sub_t["onset"].values
        X_full = sm.add_constant(sub_t[entrants_covars].values)
        countries = sub_t["ccode"].values
        unique_cc = np.unique(countries)

        # Build index lookup for fast bootstrap sampling
        cc_idx = {cc: np.where(countries == cc)[0] for cc in unique_cc}

        # Point estimate for this draw (no bootstrap)
        try:
            logit_t = _fit_logit(y_full, X_full)
            all_coefs.append(logit_t.params)
            draw_labels.append(draw_key)
        except Exception:
            pass

        # P bootstrap replications: pairs cluster bootstrap
        for p in range(P_REPS):
            # Resample countries with replacement
            cc_sample = rng.choice(unique_cc, size=len(unique_cc), replace=True)
            boot_idx = np.concatenate([cc_idx[cc] for cc in cc_sample])

            y_b = y_full[boot_idx]
            X_b = X_full[boot_idx]

            # Need variation in y for logit
            if y_b.sum() == 0 or y_b.sum() == len(y_b):
                continue

            try:
                m_b = _fit_logit(y_b, X_b, maxiter=500)
                all_coefs.append(m_b.params)
                draw_labels.append(draw_key)
            except Exception:
                continue

        if draw_idx % 5 == 0:
            print(f"  Draw {draw_idx}/{T_DRAWS} done, "
                  f"{len(all_coefs):,} coef vectors so far")

print(f"\nTotal coefficient vectors: {len(all_coefs):,} "
      f"(expected ~{T_DRAWS * (P_REPS + 1):,})")

# ── Summarise ──────────────────────────────────────────────────────────────
boot_arr = np.array(all_coefs)  # shape: (n_total, n_covars+1)
boot_df = pd.DataFrame(boot_arr, columns=coef_names)
boot_df["draw"] = draw_labels

print(f"\nT×P Bootstrap results (Entrants model):")
print(f"{'Variable':>25s}  {'Mean':>8s}  {'SD':>8s}  "
      f"{'CI_lo':>8s}  {'CI_hi':>8s}  {'Naive SE':>8s}  {'Inflation':>8s}")

# Get naive SEs for comparison (from averaged-shadow model)
naive_sub = data[list(dict.fromkeys(["onset"] + entrants_covars))].dropna()
naive_logit = _fit_logit(
    naive_sub["onset"].values,
    sm.add_constant(naive_sub[entrants_covars].values),
)

for i, var in enumerate(coef_names):
    mean_b = boot_arr[:, i].mean()
    sd_b   = boot_arr[:, i].std()
    ci_lo  = np.percentile(boot_arr[:, i], 2.5)
    ci_hi  = np.percentile(boot_arr[:, i], 97.5)
    naive_se = naive_logit.bse[i]
    inflation = sd_b / naive_se
    sig = ""
    if ci_lo > 0 or ci_hi < 0:
        sig = " *"
    print(f"{var:>25s}  {mean_b:+8.4f}  {sd_b:8.4f}  "
          f"{ci_lo:+8.4f}  {ci_hi:+8.4f}  {naive_se:8.4f}  {inflation:8.2f}{sig}")

# ── Decompose variance: measurement vs primary-analysis ────────────────────
# Following Rubin's rules: total var = within-draw var + between-draw var
print("\n── Variance decomposition (shadow vars only) ──")

for var in entrants_shadow:
    # Within-draw variance (average of per-draw variances)
    within = boot_df.groupby("draw")[var].var().mean()
    # Between-draw variance (variance of per-draw means)
    between = boot_df.groupby("draw")[var].mean().var()
    total = boot_df[var].var()
    pct_between = between / total * 100
    print(f"  {var}: total_var={total:.4f}, "
          f"within={within:.4f} ({100-pct_between:.1f}%), "
          f"between={between:.4f} ({pct_between:.1f}%)")

# ── Save ───────────────────────────────────────────────────────────────────
boot_summary = pd.DataFrame({
    "variable": coef_names,
    "mean":     boot_arr.mean(axis=0),
    "sd":       boot_arr.std(axis=0),
    "ci_lo":    np.percentile(boot_arr, 2.5, axis=0),
    "ci_hi":    np.percentile(boot_arr, 97.5, axis=0),
    "naive_se": naive_logit.bse,
    "inflation": boot_arr.std(axis=0) / naive_logit.bse,
})
boot_summary.to_parquet(RESULTS / "stage2_bootstrap.parquet", index=False)
print(f"\nBootstrap summary saved: {RESULTS / 'stage2_bootstrap.parquet'}")

T×P bootstrap: 25 draws × 200 reps = 5,000 total
Covariates: 14


  Draw 5/25 done, 1,005 coef vectors so far


  Draw 10/25 done, 2,010 coef vectors so far


  Draw 15/25 done, 3,015 coef vectors so far


  Draw 20/25 done, 4,020 coef vectors so far


/Users/rjc/portfolio/shadow/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Draw 25/25 done, 5,025 coef vectors so far

Total coefficient vectors: 5,025 (expected ~5,025)

T×P Bootstrap results (Entrants model):
                 Variable      Mean        SD     CI_lo     CI_hi  Naive SE  Inflation
                    const  -29.9606   12.6220  -56.0728   -6.6310    9.4625      1.33 *
              polity2_lag   -0.0317    0.0158   -0.0636   -0.0014    0.0139      1.14 *
                 lgdp_lag   -0.5584    0.1674   -0.9047   -0.2522    0.1227      1.36 *
                 lpop_lag   +0.0802    0.0915   -0.1060   +0.2558    0.0677      1.35
                  lmtnest   +0.1001    0.1043   -0.1194   +0.2971    0.0695      1.50
                  ncontig   +0.5647    1.3983   -0.1064   +1.1995    0.2232      6.26
                      oil   +0.3697    0.2707   -0.1553   +0.9031    0.2081      1.30
                  nwstate   +1.0745    0.4376   +0.1243   +1.8409    0.3446      1.27 *
               instab_lag   +0.5372    0.1981   +0.1428   +0.9242    0.1789    

In [ ]:
# ── T×P Bootstrap for Conditional Logit (FE Entrants) ──────────────────────
# Same procedure: 25 measurement draws × 200 cluster-bootstrap reps
# Conditional logit drops countries without onset variation in each resample

rng_fe = np.random.default_rng(seed=20260304)

fe_covars = FE_TIME_VARS + SHADOW_VARS
fe_coef_names = fe_covars  # no constant in conditional logit

print(f"T×P bootstrap (FE): {T_DRAWS} draws × {P_REPS} reps")
print(f"FE covariates: {len(fe_covars)}")

fe_all_coefs = []
fe_draw_labels = []
draw_idx = 0

for cy in range(1, 6):
    for ud in range(1, 6):
        draw_idx += 1
        draw_key = f"{cy}_{ud}"

        sh = pd.read_parquet(
            INTERIM / f"cy_shadow_{cy}_{ud}.parquet",
            columns=["ccode", "year"] + SHADOW_VARS,
        )

        df_t = base_df.merge(sh, on=["ccode", "year"], how="left")
        sub_t = df_t[list(dict.fromkeys(
            ["onset", "ccode"] + fe_covars
        ))].dropna()

        # Filter to countries with onset variation (clogit requirement)
        onset_by_cc = sub_t.groupby("ccode")["onset"].agg(["sum", "count"])
        varying = onset_by_cc[
            (onset_by_cc["sum"] > 0) & (onset_by_cc["sum"] < onset_by_cc["count"])
        ].index
        sub_v = sub_t[sub_t["ccode"].isin(varying)].copy()

        y_full = sub_v["onset"].values
        X_full = sub_v[fe_covars].astype(float).values
        countries = sub_v["ccode"].values
        unique_cc = np.unique(countries)
        cc_idx = {cc: np.where(countries == cc)[0] for cc in unique_cc}

        # Point estimate for this draw
        try:
            res_t = ConditionalLogit(y_full, X_full, groups=countries).fit(disp=False)
            fe_all_coefs.append(res_t.params)
            fe_draw_labels.append(draw_key)
        except Exception:
            pass

        # P bootstrap replications
        for p in range(P_REPS):
            cc_sample = rng_fe.choice(unique_cc, size=len(unique_cc), replace=True)
            boot_idx = np.concatenate([cc_idx[cc] for cc in cc_sample])

            y_b = y_full[boot_idx]
            X_b = X_full[boot_idx]
            g_b = countries[boot_idx]

            # Need onset variation for clogit
            onset_check = pd.Series(y_b).groupby(g_b).agg(["sum", "count"])
            if not ((onset_check["sum"] > 0) & (onset_check["sum"] < onset_check["count"])).any():
                continue

            # Filter to varying groups in this bootstrap sample
            vary_g = onset_check[
                (onset_check["sum"] > 0) & (onset_check["sum"] < onset_check["count"])
            ].index
            mask = pd.Series(g_b).isin(vary_g).values

            try:
                res_b = ConditionalLogit(y_b[mask], X_b[mask], groups=g_b[mask]).fit(disp=False)
                fe_all_coefs.append(res_b.params)
                fe_draw_labels.append(draw_key)
            except Exception:
                continue

        if draw_idx % 5 == 0:
            print(f"  Draw {draw_idx}/{T_DRAWS} done, "
                  f"{len(fe_all_coefs):,} coef vectors so far")

print(f"\nTotal FE coefficient vectors: {len(fe_all_coefs):,}")

# ── Summarise ──────────────────────────────────────────────────────────────
fe_boot_arr = np.array(fe_all_coefs)
fe_boot_df = pd.DataFrame(fe_boot_arr, columns=fe_coef_names)
fe_boot_df["draw"] = fe_draw_labels

# Naive SEs from averaged-shadow FE model for comparison
fe_naive = fe_results["FE Entrants"]

print(f"\nT×P Bootstrap results (FE Entrants model):")
print(f"{'Variable':>25s}  {'Mean':>8s}  {'SD':>8s}  "
      f"{'CI_lo':>8s}  {'CI_hi':>8s}  {'Naive SE':>8s}  {'Inflation':>8s}")

for i, var in enumerate(fe_coef_names):
    mean_b = fe_boot_arr[:, i].mean()
    sd_b   = fe_boot_arr[:, i].std()
    ci_lo  = np.percentile(fe_boot_arr[:, i], 2.5)
    ci_hi  = np.percentile(fe_boot_arr[:, i], 97.5)
    naive_se = fe_naive["res"].bse[i]
    inflation = sd_b / naive_se
    sig = ""
    if ci_lo > 0 or ci_hi < 0:
        sig = " *"
    print(f"{var:>25s}  {mean_b:+8.4f}  {sd_b:8.4f}  "
          f"{ci_lo:+8.4f}  {ci_hi:+8.4f}  {naive_se:8.4f}  {inflation:8.2f}{sig}")

# ── Variance decomposition ────────────────────────────────────────────────
print("\n── Variance decomposition (shadow vars, FE model) ──")
for var in SHADOW_VARS:
    within = fe_boot_df.groupby("draw")[var].var().mean()
    between = fe_boot_df.groupby("draw")[var].mean().var()
    total = fe_boot_df[var].var()
    pct_between = between / total * 100
    print(f"  {var}: total_var={total:.4f}, "
          f"within={within:.4f} ({100-pct_between:.1f}%), "
          f"between={between:.4f} ({pct_between:.1f}%)")

# ── Save ───────────────────────────────────────────────────────────────────
fe_boot_summary = pd.DataFrame({
    "variable": fe_coef_names,
    "mean":     fe_boot_arr.mean(axis=0),
    "sd":       fe_boot_arr.std(axis=0),
    "ci_lo":    np.percentile(fe_boot_arr, 2.5, axis=0),
    "ci_hi":    np.percentile(fe_boot_arr, 97.5, axis=0),
    "naive_se": fe_naive["res"].bse,
    "inflation": fe_boot_arr.std(axis=0) / fe_naive["res"].bse,
})
fe_boot_summary.to_parquet(RESULTS / "stage2_bootstrap_fe.parquet", index=False)
print(f"\nFE Bootstrap summary saved: {RESULTS / 'stage2_bootstrap_fe.parquet'}")

## § 10  Labeled-Only Robustness Check (KLC 2022 §5.6)

Knox, Lucas & Cho (2022, §5.6) recommend comparing full-sample estimates to
labeled-only estimates as a robustness check.  In our context:

- **Full sample** (1946–2014, N ≈ 8,792): includes both the Regan period
  (where interventions are observed and used to train the shadow) and the
  post-1999 period (where interventions are unobserved and the shadow is
  pure prediction).
- **Labeled-only** (1946–1999, N ≈ 6,516): restricted to country-years
  where the Regan intervention data exists, so the shadow is based on
  out-of-fold predictions against actual training labels.

If estimates differ markedly between samples, it may indicate that the
shadow measure behaves differently where it was trained vs. where it is
extrapolating.  Similar estimates strengthen the case that the shadow
captures a genuine structural relationship rather than in-sample artifacts.

In [ ]:
# ── §5.6 Labeled-only robustness check (KLC 2022) ──────────────────────────
# Compare Entrants model on full sample vs. Regan-period only

print("=" * 60)
print("§ 10  Labeled-Only Robustness Check (KLC §5.6)")
print("=" * 60)

entrants_shadow = ["E_gov_trim_asinh", "E_opp_trim_asinh"]
entrants_covars = BASELINE_VARS + entrants_shadow

# Full sample (already computed in §3, but redo for clarity)
cols_needed = list(dict.fromkeys(["onset", "year", "cv_group", "ccode"] + entrants_covars))
sub_full = data[cols_needed].dropna().copy()
y_full = sub_full["onset"].values
X_full = sm.add_constant(sub_full[entrants_covars].values)
logit_full = _fit_logit(y_full, X_full)

# Labeled-only: restrict to year <= 1999 (Regan intervention coding window)
sub_label = sub_full[sub_full["year"] <= 1999].copy()
y_label = sub_label["onset"].values
X_label = sm.add_constant(sub_label[entrants_covars].values)
logit_label = _fit_logit(y_label, X_label)

# Compare coefficients
coef_names = ["const"] + entrants_covars
print(f"\n{'Variable':>25s}  {'Full (1946-2014)':>18s}  {'Labeled (1946-99)':>18s}  {'Diff':>8s}")
print("-" * 80)
for i, var in enumerate(coef_names):
    c_f = logit_full.params[i]
    se_f = logit_full.bse[i]
    c_l = logit_label.params[i]
    se_l = logit_label.bse[i]
    diff = c_f - c_l
    print(f"{var:>25s}  {c_f:+7.4f} ({se_f:.4f})  {c_l:+7.4f} ({se_l:.4f})  {diff:+7.4f}")

# Fit metrics comparison
print(f"\n{'Metric':>20s}  {'Full':>10s}  {'Labeled':>10s}")
print("-" * 45)
print(f"{'N':>20s}  {len(y_full):>10,}  {len(y_label):>10,}")
print(f"{'Onsets':>20s}  {int(y_full.sum()):>10}  {int(y_label.sum()):>10}")

is_prl_full = prl(y_full, logit_full.predict(X_full))
is_prl_label = prl(y_label, logit_label.predict(X_label))
is_auc_full = roc_auc_score(y_full, logit_full.predict(X_full))
is_auc_label = roc_auc_score(y_label, logit_label.predict(X_label))
print(f"{'IS PRL':>20s}  {is_prl_full:>10.3f}  {is_prl_label:>10.3f}")
print(f"{'IS AUC':>20s}  {is_auc_full:>10.3f}  {is_auc_label:>10.3f}")

# OOS comparison (leave-one-onset-group-out within each sample)
from sklearn.model_selection import LeaveOneGroupOut

for sample_name, sub, logit_model in [
    ("Full", sub_full, logit_full),
    ("Labeled", sub_label, logit_label),
]:
    y_s = sub["onset"].values
    X_s = sm.add_constant(sub[entrants_covars].values)
    groups_s = sub["cv_group"].values
    logo = LeaveOneGroupOut()
    oof = np.zeros(len(y_s))
    for tr_idx, val_idx in logo.split(X_s, y_s, groups_s):
        try:
            m_fold = _fit_logit(y_s[tr_idx], X_s[tr_idx], maxiter=500)
            oof[val_idx] = m_fold.predict(X_s[val_idx])
        except Exception:
            oof[val_idx] = y_s[tr_idx].mean()
    oof_clipped = np.clip(oof, 1e-9, 1 - 1e-9)
    oos_prl_s = prl(y_s, oof_clipped)
    oos_auc_s = roc_auc_score(y_s, oof)
    print(f"{'OOS PRL (' + sample_name + ')':>20s}  {oos_prl_s:>10.3f}")
    print(f"{'OOS AUC (' + sample_name + ')':>20s}  {oos_auc_s:>10.3f}")

# Save
labeled_check = pd.DataFrame({
    "variable": coef_names,
    "coef_full": logit_full.params,
    "se_full": logit_full.bse,
    "coef_labeled": logit_label.params,
    "se_labeled": logit_label.bse,
    "diff": logit_full.params - logit_label.params,
})
labeled_check.to_parquet(RESULTS / "stage2_labeled_only.parquet", index=False)
print(f"\nLabeled-only comparison saved: {RESULTS / 'stage2_labeled_only.parquet'}")